# SmartMentor · Prueba "Mago de Oz" con Gemini + embeddings

Este notebook **es el mago detrás de la cortina**: le pide a Gemini, con prompts versionados, todo lo que la plataforma mostrará, y deja escrito el contrato de datos entre la IA y la interfaz.

**Qué produce** (todo queda en `mago_de_oz_out/`, listo para copiar al repositorio):

| Carpeta | Contenido |
|---|---|
| `schemas/` | Esquemas JSON versionados de cada salida: syllabus, diagnóstico, objetivo, ruta, explicación del tema, video, quiz, calificación, evaluación final y badge |
| `prompts/` | Cada prompt con su versión (`explicacion_tema v1.0`, …) |
| `salidas/` | Salidas reales del LLM + `fuente_unica_laura.json` (un solo conjunto de datos) + datos por pantalla |
| `llm_log.jsonl` | Cada llamada: tarea, modelo, versión de prompt, tokens, latencia, costo, si el JSON validó |

**Flujo**: syllabus → estructuración (LLM) → validación del docente (DAG) → diagnóstico (LLM) + perfil semántico (embeddings) → objetivo (LLM) → ruta (algoritmo determinista) → tema completo con RAG (embeddings + LLM), video re-ordenado por embeddings, quiz y calificación (LLM-as-judge) → evaluación final → badge Open Badges.

La estudiante "Laura" es ficticia y sus respuestas las produce un **estudiante simulado con LLM** (el brief lo propone como respaldo cuando hay pocos participantes). Las respuestas del piloto real reemplazan esa parte.

**Requisitos**: una API key de Gemini ([Google AI Studio](https://aistudio.google.com/apikey)) en la variable `GEMINI_API_KEY` (en Colab: *Secrets* → `GEMINI_API_KEY`). Opcional: `YOUTUBE_API_KEY` para buscar videos.

In [ ]:
%pip install -q -U google-genai pydantic jsonschema numpy pandas requests

## 1. Configuración
Cambia el modelo aquí. La celda siguiente lista los modelos disponibles para tu clave, por si quieres otro (por ejemplo uno más nuevo de la familia Flash o un Pro).

In [ ]:
import os, json, time, hashlib, datetime as dt, pathlib, re, copy, textwrap
from typing import Literal, Optional
import numpy as np
import pandas as pd
from pydantic import BaseModel, Field, ConfigDict, ValidationError
from google import genai
from google.genai import types

# --- API key: Colab Secrets o variable de entorno ---
if not os.environ.get("GEMINI_API_KEY"):
    try:
        from google.colab import userdata
        os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
    except Exception:
        pass
assert os.environ.get("GEMINI_API_KEY"), "Define GEMINI_API_KEY (variable de entorno o Colab Secrets)."

client = genai.Client()  # toma GEMINI_API_KEY del entorno

# --- Modelos (cámbialos aquí) ---
LLM_MODEL   = os.environ.get("SM_LLM_MODEL", "gemini-3.5-flash")      # generación
EMBED_MODEL = os.environ.get("SM_EMBED_MODEL", "gemini-embedding-001")  # embeddings de texto
EMBED_DIM   = 768          # dimensión reducida (gemini-embedding-001 admite 768/1536/3072)
TEMPERATURE = 0.4

# --- Reglas de negocio del brief (las define el docente por ruta) ---
UMBRAL_TEMA    = 0.70   # aprobar quiz de tema
UMBRAL_FINAL   = 0.80   # evaluación final → badge
UMBRAL_DOMINIO = 0.80   # tema omitido si el diagnóstico ya muestra dominio
MAX_INTENTOS   = 3

# --- Precios para el log de costo (USD por 1M tokens). Llénalos desde ai.google.dev/gemini-api/docs/pricing ---
PRECIOS_USD_POR_MILLON = {
    # "gemini-3.5-flash": {"entrada": 0.0, "salida": 0.0},
    # "gemini-embedding-001": {"entrada": 0.0, "salida": 0.0},
}

REGENERAR = False   # True = ignora la caché y vuelve a llamar al LLM

OUT = pathlib.Path("mago_de_oz_out")
for d in ["schemas", "prompts", "salidas", "cache"]:
    (OUT / d).mkdir(parents=True, exist_ok=True)
LOG_PATH = OUT / "llm_log.jsonl"
print("LLM:", LLM_MODEL, "| Embeddings:", EMBED_MODEL, "| salida en", OUT.resolve())

In [ ]:
# Opcional: modelos disponibles para tu clave
try:
    filas = []
    for m in client.models.list():
        acciones = getattr(m, "supported_actions", None) or []
        if "generateContent" in acciones or "embedContent" in acciones:
            filas.append({"modelo": m.name.replace("models/", ""), "acciones": ", ".join(acciones)})
    display(pd.DataFrame(filas))
except Exception as e:
    print("No se pudo listar modelos:", e)

## 2. Contratos de datos (esquemas JSON)

Este es el entregable principal de la actividad. Cada salida de IA tiene dos capas:

- **Modelo `...LLM`**: lo que se le exige al LLM devolver (se pasa como `response_json_schema`).
- **Contrato**: el modelo LLM + metadatos de trazabilidad (`meta.prompt_version`, modelo, hash de entrada). Es lo que guarda el backend y lo que recibe la pantalla.

Los esquemas evitan diccionarios libres y uniones complejas en la parte que va al LLM, porque son las construcciones que Gemini respeta peor; los rangos (0–1, índices) se validan en código.

In [ ]:
Bloom = Literal["Recordar", "Comprender", "Aplicar", "Analizar", "Evaluar", "Crear"]
ORDEN_BLOOM = ["Recordar", "Comprender", "Aplicar", "Analizar", "Evaluar", "Crear"]

class Meta(BaseModel):
    prompt_id: str
    prompt_version: str
    modelo: str
    generado_en: str
    entrada_sha256: str
    desde_cache: bool = False

# ---------- Syllabus ----------
class ResultadoAprendizaje(BaseModel):
    id: str = Field(description="id corto, ej. ra_regresion_logistica_1")
    enunciado: str = Field(description="Verbo de Bloom + contenido + criterio medible")
    nivel_bloom: Bloom

class TemaSyllabus(BaseModel):
    id: str = Field(description="snake_case, estable, ej. regresion_logistica")
    nombre: str
    descripcion: str
    subtemas: list[str]
    resultados_aprendizaje: list[ResultadoAprendizaje]
    nivel_bloom: Bloom = Field(description="Nivel más alto entre sus resultados de aprendizaje")
    prerrequisitos: list[str] = Field(description="ids de temas de este mismo syllabus")
    horas_estimadas: int

class SyllabusLLM(BaseModel):
    curso_id: str
    curso_nombre: str
    objetivo_por_defecto: str
    temas: list[TemaSyllabus]

class Syllabus(SyllabusLLM):
    aprobado_por_docente: bool = False
    meta: Meta

# ---------- Diagnóstico ----------
class PreguntaOM(BaseModel):
    id: str
    tema_id: str
    enunciado: str
    bloom: Bloom
    dificultad: int = Field(description="1 = Recordar, 2 = Aplicar, 3 = Analizar")
    opciones: list[str]
    respuesta_correcta: int = Field(description="Índice 0-based de la opción correcta")
    justificacion: str

class DiagnosticoLLM(BaseModel):
    preguntas: list[PreguntaOM]

class RespuestaEstudiante(BaseModel):
    pregunta_id: str
    opcion_elegida: int = Field(description="Índice elegido; -1 si es respuesta corta")
    texto: str = Field(description="Respuesta escrita; vacío en opción múltiple")

class RespuestasSimuladas(BaseModel):
    respuestas: list[RespuestaEstudiante]

class NivelTema(BaseModel):
    tema_id: str
    puntaje_prueba: float          # aciertos ponderados por dificultad, 0-1
    similitud_perfil: float        # coseno perfil vs tema
    prior_perfil: float            # similitud normalizada, 0-1
    peso_perfil: float             # cuánto pesa el perfil (0 si escribió poco)
    nivel_combinado: float         # 0-1
    dominado: bool

class Diagnostico(BaseModel):
    inscripcion_id: str
    preguntas: list[PreguntaOM]
    respuestas: list[RespuestaEstudiante]
    nivel_por_tema: list[NivelTema]
    puntaje_global: float
    meta_preguntas: Meta
    meta_respuestas: Meta

# ---------- Objetivo ----------
class ObjetivoLLM(BaseModel):
    enunciado: str = Field(description="Objetivo completo: verbo de Bloom + contenido + criterio")
    verbo_bloom: str
    contenido: str
    criterio: str
    temas_foco: list[str] = Field(description="ids de temas de la brecha")
    justificacion: str

class Objetivo(ObjetivoLLM):
    inscripcion_id: str
    origen: Literal["ruta", "propuesto_llm"]
    confirmado_por_estudiante: bool
    meta: Optional[Meta] = None

# ---------- Ruta ----------
class PasoRuta(BaseModel):
    orden: int
    tema_id: str
    nombre: str
    incluido: bool
    motivo: str
    estado: Literal["pendiente", "en_curso", "aprobado", "omitido", "bloqueado"]
    puntaje: Optional[float] = None
    intentos: int = 0

class RutaPersonalizada(BaseModel):
    inscripcion_id: str
    ruta_id: str
    algoritmo: str
    umbral_dominio: float
    pasos: list[PasoRuta]

# ---------- Explicación del tema (RAG) ----------
class Seccion(BaseModel):
    subtitulo: str
    contenido: str = Field(description="Texto con citas [n] a los fragmentos")

class ExplicacionLLM(BaseModel):
    titulo: str
    secciones: list[Seccion]
    ejemplo: str
    resumen: str
    preguntas_reflexion: list[str]
    vacios_detectados: list[str] = Field(description="Resultados de aprendizaje que los fragmentos no cubren")

class FuenteCitada(BaseModel):
    n: int
    fuente_id: str
    titulo: str
    url: str

class ExplicacionTema(ExplicacionLLM):
    tema_id: str
    tipo: Literal["principal", "refuerzo"]
    fuentes: list[FuenteCitada]
    fidelidad: Optional[float] = None
    meta: Meta

class Afirmacion(BaseModel):
    texto: str
    citas: list[int]
    soportada: bool
    comentario: str

class VerificacionFidelidad(BaseModel):
    afirmaciones: list[Afirmacion]

# ---------- Video ----------
class VideoCandidato(BaseModel):
    video_id: str
    titulo: str
    url: str
    canal: str = ""
    duracion_min: Optional[float] = None
    similitud: Optional[float] = None

class RecursoVideo(BaseModel):
    tema_id: str
    origen: Literal["youtube_api", "lista_docente", "ninguno"]
    candidatos: list[VideoCandidato]
    seleccionado: Optional[VideoCandidato] = None
    embed_model: str

# ---------- Quiz y evaluación final ----------
class CriterioRubrica(BaseModel):
    criterio: str
    puntos: float

class PreguntaQuiz(BaseModel):
    id: str
    tema_id: str
    resultado_id: str = Field(description="Resultado de aprendizaje que evalúa")
    enunciado: str
    tipo: Literal["opcion_multiple", "respuesta_corta"]
    bloom: Bloom
    opciones: list[str] = Field(description="Vacío si es respuesta corta")
    respuesta_correcta: int = Field(description="Índice 0-based; -1 si es respuesta corta")
    respuesta_modelo: str = Field(description="Justificación (OM) o respuesta esperada (respuesta corta)")
    rubrica: list[CriterioRubrica] = Field(description="Solo respuesta corta; vacío en OM")
    fuentes: list[int] = Field(description="Números [n] de fragmentos que respaldan la respuesta")

class QuizLLM(BaseModel):
    preguntas: list[PreguntaQuiz]

class Quiz(QuizLLM):
    quiz_id: str
    tema_id: str
    intento: int
    umbral_aprobacion: float
    meta: Meta

class EvaluacionFinal(QuizLLM):
    evaluacion_id: str
    temas_evaluados: list[str]
    intento: int
    umbral_aprobacion: float
    meta: Meta

# ---------- Calificación ----------
class CriterioEvaluado(BaseModel):
    criterio: str
    cumplido: bool
    puntos_otorgados: float
    evidencia: str

class CalificacionLLM(BaseModel):
    criterios: list[CriterioEvaluado]
    retroalimentacion: str

class ResultadoPregunta(BaseModel):
    pregunta_id: str
    tema_id: str
    tipo: str
    correcta: Optional[bool] = None
    puntaje: float                       # 0-1
    retroalimentacion: str
    calificador: Literal["regla", "llm_juez"]
    detalle_rubrica: list[CriterioEvaluado] = []

class Calificacion(BaseModel):
    intento_id: str
    evaluacion_id: str
    tipo: Literal["tema", "final"]
    tema_id: Optional[str] = None
    respuestas: list[RespuestaEstudiante]
    por_pregunta: list[ResultadoPregunta]
    puntaje_total: float
    aprobado: bool
    fecha: str
    meta_respuestas: Meta
    meta_calificador: list[Meta]

# ---------- Badge (Open Badges 2.0) ----------
class BadgeTextoLLM(BaseModel):
    nombre: str
    descripcion: str
    criterios: str
    competencias: list[str]

class OBCriteria(BaseModel):
    narrative: str

class OBIssuer(BaseModel):
    type: str = "Profile"
    id: str
    name: str
    url: str

class OBBadgeClass(BaseModel):
    type: str = "BadgeClass"
    id: str
    name: str
    description: str
    image: str
    criteria: OBCriteria
    issuer: OBIssuer
    tags: list[str]

class OBRecipient(BaseModel):
    type: str = "email"
    hashed: bool = True
    salt: str
    identity: str

class OBVerification(BaseModel):
    type: str = "hosted"

class OBEvidence(BaseModel):
    narrative: str

class BadgeAssertion(BaseModel):
    model_config = ConfigDict(populate_by_name=True)
    context: str = Field("https://w3id.org/openbadges/v2", alias="@context")
    type: str = "Assertion"
    id: str
    recipient: OBRecipient
    badge: OBBadgeClass
    issuedOn: str
    verification: OBVerification = OBVerification()
    evidence: list[OBEvidence]

CONTRATOS = {
    "syllabus": Syllabus, "diagnostico": Diagnostico, "objetivo": Objetivo,
    "ruta": RutaPersonalizada, "explicacion_tema": ExplicacionTema, "video": RecursoVideo,
    "quiz": Quiz, "calificacion": Calificacion, "evaluacion_final": EvaluacionFinal,
    "badge": BadgeAssertion,
}
for nombre, modelo in CONTRATOS.items():
    esquema = modelo.model_json_schema(by_alias=True)
    esquema["$id"] = f"smartmentor/schemas/{nombre}.v1.json"
    (OUT / "schemas" / f"{nombre}.v1.json").write_text(json.dumps(esquema, ensure_ascii=False, indent=2), encoding="utf-8")
print("Esquemas exportados:", sorted(p.name for p in (OUT / "schemas").glob("*.json")))

## 3. Prompts versionados
Cambiar un prompt = subir su versión. La versión viaja en `meta` de cada salida y en el log, así cada pantalla puede mostrar qué prompt produjo su contenido.

In [ ]:
PROMPTS = {
"syllabus_estructurado": {"version": "v1.0",
 "system": "Eres un diseñador instruccional experto. Extraes la estructura de un syllabus universitario sin inventar contenidos que no estén en el texto. Respondes solo con JSON que cumple el esquema.",
 "user": """Estructura el siguiente syllabus.
Reglas:
- Un tema por unidad o bloque temático del syllabus. No agregues temas que no aparezcan.
- `id` en snake_case, corto y estable (ejemplo: regresion_logistica).
- De 2 a 4 resultados de aprendizaje por tema, redactados como verbo de Bloom + contenido + criterio medible. id de cada resultado: ra_<tema_id>_<n>.
- `nivel_bloom` del tema = el nivel más alto entre sus resultados.
- `prerrequisitos` = ids de temas de ESTE syllabus que deben dominarse antes. Sin ciclos.
- `objetivo_por_defecto`: el objetivo general del curso como resultado medible.
- `curso_id` en snake_case.

Syllabus:
<<<
{syllabus}
>>>"""},

"diagnostico": {"version": "v1.0",
 "system": "Eres un experto en evaluación educativa y psicometría. Escribes ítems claros, con una sola respuesta correcta y distractores plausibles basados en errores frecuentes.",
 "user": """Genera una prueba diagnóstica para estimar el nivel previo de un estudiante en cada tema.
Para CADA tema escribe exactamente 3 preguntas de opción múltiple:
- una de nivel "Recordar" con dificultad 1,
- una de nivel "Aplicar" con dificultad 2,
- una de nivel "Analizar" con dificultad 3.
Cada pregunta: 4 opciones, una sola correcta, `respuesta_correcta` = índice 0-based. Varía la posición de la correcta. id = d_<tema_id>_<n> con n = 1, 2, 3.
No uses "todas las anteriores" ni "ninguna de las anteriores".

Temas (JSON):
{temas}"""},

"estudiante_simulado": {"version": "v1.0",
 "system": "Simulas a un estudiante real para una prueba de usabilidad. Nunca respondes como un experto si el perfil no lo justifica.",
 "user": """Perfil del estudiante (en sus palabras):
\"\"\"{perfil}\"\"\"

Lo que ha estudiado en la plataforma hasta ahora:
\"\"\"{contexto}\"\"\"

Responde las preguntas como lo haría ESTA persona: acierta lo que su formación y lo estudiado le permiten saber, y equivócate de forma realista en lo que no domina (no aciertes todo por defecto).
- Opción múltiple: `opcion_elegida` = índice 0-based y `texto` vacío.
- Respuesta corta: `opcion_elegida` = -1 y `texto` con 1 a 3 frases en el estilo de la persona.
Responde todas las preguntas, en el mismo orden, usando su `id` como `pregunta_id`.

Preguntas (JSON):
{preguntas}"""},

"objetivo_aprendizaje": {"version": "v1.0",
 "system": "Eres un tutor que redacta objetivos de aprendizaje medibles según la taxonomía de Bloom.",
 "user": """Propón el objetivo de aprendizaje personal de este estudiante para la ruta.
- Debe ser coherente con el objetivo de la ruta y centrarse en su brecha.
- Formato: verbo de Bloom + contenido + criterio medible (ej. "con al menos 80 % en la evaluación final").
- `temas_foco`: ids de los temas de la brecha (solo ids de la lista).

Objetivo de la ruta: {objetivo_ruta}
Perfil del estudiante: \"\"\"{perfil}\"\"\"
Nivel por tema (0 = nada, 1 = domina): {brecha}"""},

"explicacion_tema": {"version": "v1.0",
 "system": "Eres un tutor universitario. Explicas usando SOLO la información de los fragmentos que se te entregan y citas cada afirmación con [n]. Haces pensar al estudiante en vez de darle respuestas listas.",
 "user": """Explica el tema para este estudiante.
Tema (JSON): {tema}
Nivel estimado del estudiante en este tema: {nivel} (0 a 1).
Perfil: \"\"\"{perfil}\"\"\"
{modo}

Reglas:
- Usa SOLO los fragmentos. Cada afirmación factual lleva su cita [n]. No inventes fuentes ni números de cita.
- 3 a 5 secciones que cubran los resultados de aprendizaje, en un tono claro y en español.
- `ejemplo`: un caso práctico conectado con el perfil del estudiante.
- `preguntas_reflexion`: 2 preguntas que lo hagan razonar (sin respuesta).
- `vacios_detectados`: resultados de aprendizaje que los fragmentos no cubren (lista vacía si no hay).

Fragmentos:
{fragmentos}"""},

"fidelidad_rag": {"version": "v1.0",
 "system": "Eres un verificador de hechos estricto.",
 "user": """Descompón el texto en afirmaciones factuales atómicas (máximo 15). Para cada una indica las citas [n] que usa y si los fragmentos citados la soportan de verdad (`soportada`). Ignora frases que no son factuales.

Texto:
{texto}

Fragmentos:
{fragmentos}"""},

"quiz_tema": {"version": "v1.0",
 "system": "Eres un experto en evaluación. Generas preguntas desde los resultados de aprendizaje, no desde un texto, para no evaluar solo memorización.",
 "user": """Genera el quiz del tema (intento {intento}).
- {n_om} preguntas de opción múltiple (4 opciones, una correcta, `rubrica` vacía) y {n_rc} de respuesta corta (`opciones` vacío, `respuesta_correcta` = -1, rúbrica de 2 a 3 criterios cuyos puntos suman 1).
- Cubre todos los resultados de aprendizaje; `resultado_id` dice cuál evalúa cada pregunta. Mezcla niveles de Bloom.
- `tema_id` = "{tema_id}". id = q<n>.
- Las preguntas deben ser DISTINTAS a las de intentos anteriores.
- `fuentes`: números [n] de los fragmentos que respaldan la respuesta correcta (lista vacía si ninguno).

Tema (JSON): {tema}
Preguntas de intentos anteriores (no repetir): {previas}
Fragmentos de referencia:
{fragmentos}"""},

"calificacion_respuesta_corta": {"version": "v1.0",
 "system": "Eres un calificador justo y consistente. Aplicas la rúbrica literalmente, sin premiar la extensión ni penalizar la redacción.",
 "user": """Califica la respuesta del estudiante con la rúbrica. Para cada criterio: `cumplido`, `puntos_otorgados` (entre 0 y los puntos del criterio; puedes dar puntaje parcial) y `evidencia` (la parte de la respuesta que lo justifica). Luego una retroalimentación breve que oriente sin dar la respuesta completa.

Pregunta: {pregunta}
Respuesta esperada: {respuesta_modelo}
Rúbrica (JSON): {rubrica}
Respuesta del estudiante: \"\"\"{respuesta}\"\"\""""},

"evaluacion_final": {"version": "v1.0",
 "system": "Eres un experto en evaluación. Diseñas evaluaciones sumativas alineadas con los resultados de aprendizaje.",
 "user": """Genera la evaluación final de la ruta (intento {intento}).
- Por cada tema: 2 preguntas de opción múltiple (4 opciones, una correcta, `rubrica` vacía), una de nivel Comprender o Aplicar y otra de nivel Analizar.
- Además {n_rc} preguntas de respuesta corta que integren varios temas (`opciones` vacío, `respuesta_correcta` = -1, rúbrica de 2 a 3 criterios que suman 1; usa como `tema_id` el tema principal).
- `resultado_id` = resultado de aprendizaje evaluado. id = f<n>. `fuentes` = lista vacía.
- Preguntas distintas a las de intentos anteriores: {previas}

Temas de la ruta (JSON):
{temas}"""},

"badge_texto": {"version": "v1.0",
 "system": "Redactas credenciales digitales sobrias y verificables.",
 "user": """Redacta el texto del badge (en español): nombre corto, descripción de 1 a 2 frases, criterios de obtención (umbral y evaluaciones) y de 3 a 5 competencias demostradas.
Curso: {curso}
Objetivo logrado: {objetivo}
Temas de la ruta: {temas}
Umbral de la evaluación final: {umbral}"""},
}

for pid, p in PROMPTS.items():
    (OUT / "prompts" / f"{pid}.{p['version']}.json").write_text(
        json.dumps({"id": pid, **p}, ensure_ascii=False, indent=2), encoding="utf-8")
print(len(PROMPTS), "prompts versionados en", OUT / "prompts")

## 4. Capa de IA: llamada estructurada a Gemini, embeddings, caché y log

- `llm_json` exige el esquema con `response_json_schema`, valida con Pydantic, reintenta si el JSON no valida (y lo registra: es la métrica "tasa de JSON inválido") y guarda en caché por hash de entrada.
- `embed` usa `task_type` distinto para documentos y consultas, y normaliza los vectores (necesario al reducir dimensión).

In [ ]:
def _ahora():
    return dt.datetime.now(dt.timezone.utc).isoformat(timespec="seconds")

def _sha(texto: str) -> str:
    return hashlib.sha256(texto.encode("utf-8")).hexdigest()

def esquema_para_gemini(modelo: type[BaseModel]) -> dict:
    """JSON Schema con las referencias $defs resueltas en línea."""
    s = modelo.model_json_schema()
    defs = s.pop("$defs", {})
    def resolver(x):
        if isinstance(x, dict):
            if "$ref" in x:
                return resolver(copy.deepcopy(defs[x["$ref"].split("/")[-1]]))
            return {k: resolver(v) for k, v in x.items()}
        if isinstance(x, list):
            return [resolver(i) for i in x]
        return x
    return resolver(s)

def _costo(modelo, t_in, t_out):
    p = PRECIOS_USD_POR_MILLON.get(modelo)
    return None if not p else (t_in * p["entrada"] + t_out * p["salida"]) / 1e6

def _log(fila: dict):
    with open(LOG_PATH, "a", encoding="utf-8") as f:
        f.write(json.dumps(fila, ensure_ascii=False) + "\n")

def llm_json(tarea: str, esquema: type[BaseModel], variables: dict,
             temperature: float | None = None, max_reintentos: int = 2):
    """Llama a Gemini con salida estructurada. Devuelve (objeto_validado, Meta)."""
    p = PROMPTS[tarea]
    prompt = p["user"].format(**variables)
    clave = _sha(json.dumps([LLM_MODEL, tarea, p["version"], p["system"], prompt,
                             esquema.__name__], ensure_ascii=False))
    ruta_cache = OUT / "cache" / f"{tarea}_{clave[:16]}.json"
    meta = dict(prompt_id=tarea, prompt_version=f"{tarea} {p['version']}", modelo=LLM_MODEL,
                generado_en=_ahora(), entrada_sha256=clave)

    if ruta_cache.exists() and not REGENERAR:
        guardado = json.loads(ruta_cache.read_text(encoding="utf-8"))
        meta.update(generado_en=guardado["generado_en"], desde_cache=True)
        return esquema.model_validate(guardado["salida"]), Meta(**meta)

    contenido = prompt
    for intento in range(max_reintentos + 1):
        t0 = time.perf_counter()
        error = None
        try:
            resp = client.models.generate_content(
                model=LLM_MODEL,
                contents=contenido,
                config=types.GenerateContentConfig(
                    system_instruction=p["system"],
                    temperature=TEMPERATURE if temperature is None else temperature,
                    response_mime_type="application/json",
                    response_json_schema=esquema_para_gemini(esquema),
                ),
            )
            texto = resp.text or ""
            uso = resp.usage_metadata
        except Exception as e:      # error de red/API: se registra y se reintenta
            texto, uso, error = "", None, f"API: {e}"
        latencia = time.perf_counter() - t0
        t_in = (getattr(uso, "prompt_token_count", 0) or 0) if uso else 0
        t_out = ((getattr(uso, "candidates_token_count", 0) or 0) +
                 (getattr(uso, "thoughts_token_count", 0) or 0)) if uso else 0
        obj = None
        if error is None:
            try:
                obj = esquema.model_validate_json(texto)
            except ValidationError as e:
                error = f"JSON inválido: {str(e)[:400]}"
        _log({"ts": _ahora(), "tarea": tarea, "prompt_version": meta["prompt_version"],
              "modelo": LLM_MODEL, "intento_llamada": intento + 1, "json_valido": obj is not None,
              "error": error, "tokens_entrada": t_in, "tokens_salida": t_out,
              "latencia_s": round(latencia, 2), "costo_usd": _costo(LLM_MODEL, t_in, t_out),
              "entrada_sha256": clave})
        if obj is not None:
            ruta_cache.write_text(json.dumps({"generado_en": meta["generado_en"], "tarea": tarea,
                                              "prompt": prompt, "salida": obj.model_dump()},
                                             ensure_ascii=False, indent=2), encoding="utf-8")
            return obj, Meta(**meta)
        print(f"  ⚠️ {tarea} intento {intento + 1}: {error[:160]}")
        if error.startswith("JSON"):
            contenido = prompt + f"\n\nTu respuesta anterior no cumplió el esquema ({error[:300]}). Devuelve solo JSON válido."
        time.sleep(2 * (intento + 1))
    raise RuntimeError(f"{tarea}: no se obtuvo una salida válida tras {max_reintentos + 1} intentos")

def embed(textos: list[str], task_type: str = "SEMANTIC_SIMILARITY") -> np.ndarray:
    """Embeddings normalizados (L2) con caché en disco."""
    clave = _sha(json.dumps([EMBED_MODEL, EMBED_DIM, task_type, textos], ensure_ascii=False))
    ruta_cache = OUT / "cache" / f"emb_{clave[:16]}.npy"
    if ruta_cache.exists() and not REGENERAR:
        return np.load(ruta_cache)
    vectores = []
    t0 = time.perf_counter()
    for i in range(0, len(textos), 100):
        r = client.models.embed_content(
            model=EMBED_MODEL, contents=textos[i:i + 100],
            config=types.EmbedContentConfig(task_type=task_type, output_dimensionality=EMBED_DIM))
        vectores += [e.values for e in r.embeddings]
    V = np.array(vectores, dtype=np.float32)
    V /= np.linalg.norm(V, axis=1, keepdims=True) + 1e-12
    tokens_aprox = sum(len(t) for t in textos) // 4
    _log({"ts": _ahora(), "tarea": f"embeddings:{task_type}", "prompt_version": None,
          "modelo": EMBED_MODEL, "intento_llamada": 1, "json_valido": True, "error": None,
          "tokens_entrada": tokens_aprox, "tokens_salida": 0,
          "latencia_s": round(time.perf_counter() - t0, 2),
          "costo_usd": _costo(EMBED_MODEL, tokens_aprox, 0), "entrada_sha256": clave})
    np.save(ruta_cache, V)
    return V

def guardar(nombre: str, obj):
    data = obj.model_dump(by_alias=True) if isinstance(obj, BaseModel) else obj
    (OUT / "salidas" / f"{nombre}.json").write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding="utf-8")

def ver(obj, n=1500):
    data = obj.model_dump(by_alias=True) if isinstance(obj, BaseModel) else obj
    s = json.dumps(data, ensure_ascii=False, indent=2)
    print(s[:n] + ("\n… (recortado)" if len(s) > n else ""))

# Prueba rápida de conexión
_r = client.models.generate_content(model=LLM_MODEL, contents="Responde solo: OK")
print("Gemini responde:", (_r.text or "").strip()[:40])

## 5. Insumos: syllabus, fuentes para RAG y perfil de la estudiante

Reemplaza el syllabus por el real (texto o archivo `syllabus.txt` / `.md` / `.pdf` junto al notebook). Las fuentes de ejemplo son apuntes breves del docente; para el piloto, pon libros abiertos o apuntes reales en la carpeta `fuentes/` (`.txt` o `.md`) y se indexan solos.

In [ ]:
SYLLABUS_TEXTO = """
MAESTRÍA EN CIENCIA DE DATOS — Curso: Introducción al Aprendizaje Automático (ruta corta, 4 semanas)

Objetivo general: que el estudiante construya, evalúe y compare modelos supervisados básicos sobre datos tabulares
y justifique la elección del modelo con métricas adecuadas.

Unidad 1. Fundamentos y preparación de datos. Aprendizaje supervisado vs. no supervisado, variables y etiquetas,
partición entrenamiento/prueba, fuga de datos, escalado de variables y codificación de variables categóricas.

Unidad 2. Regresión lineal. Modelo lineal, mínimos cuadrados, interpretación de coeficientes, error cuadrático medio
y R². Requiere la Unidad 1.

Unidad 3. Regresión logística. Clasificación binaria, función sigmoide, probabilidad y umbral de decisión,
log-loss, interpretación de coeficientes como log-odds y regularización L2. Requiere las Unidades 1 y 2.

Unidad 4. Evaluación de modelos de clasificación. Matriz de confusión, precisión, recall, F1, curva ROC y AUC,
validación cruzada k-fold y desbalance de clases. Requiere la Unidad 3.

Unidad 5. Árboles de decisión y ensambles. Criterio de división (Gini, entropía), profundidad y sobreajuste,
bosques aleatorios e importancia de variables. Requiere las Unidades 1 y 4.

Evaluación: quices por unidad (≥ 70 %) y evaluación final integradora (≥ 80 % para la insignia).
"""

for candidato in ["syllabus.txt", "syllabus.md", "syllabus.pdf"]:
    if pathlib.Path(candidato).exists():
        if candidato.endswith(".pdf"):
            import subprocess, sys
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pypdf"], check=True)
            from pypdf import PdfReader
            SYLLABUS_TEXTO = "\n".join(pg.extract_text() or "" for pg in PdfReader(candidato).pages)
        else:
            SYLLABUS_TEXTO = pathlib.Path(candidato).read_text(encoding="utf-8")
        print("Syllabus cargado desde", candidato)
        break

# Fuentes de ejemplo (apuntes del docente). url = referencia para ampliar.
FUENTES = [
 {"fuente_id": "apuntes_u1_a", "titulo": "Apuntes U1: aprendizaje supervisado", "url": "https://scikit-learn.org/stable/supervised_learning.html",
  "texto": "En el aprendizaje supervisado cada ejemplo tiene variables de entrada y una etiqueta conocida; el modelo aprende una función que asigna entradas a etiquetas. En el no supervisado no hay etiquetas y se buscan estructuras, como grupos. Si la etiqueta es numérica continua el problema es de regresión; si es una categoría, de clasificación."},
 {"fuente_id": "apuntes_u1_b", "titulo": "Apuntes U1: partición y fuga de datos", "url": "https://scikit-learn.org/stable/common_pitfalls.html",
  "texto": "Los datos se dividen en entrenamiento y prueba para estimar el desempeño en datos no vistos. La fuga de datos ocurre cuando información del conjunto de prueba o del futuro influye en el entrenamiento, por ejemplo al escalar con la media de todo el dataset antes de partir. Todo preprocesamiento debe ajustarse solo con el conjunto de entrenamiento."},
 {"fuente_id": "apuntes_u1_c", "titulo": "Apuntes U1: escalado y codificación", "url": "https://scikit-learn.org/stable/modules/preprocessing.html",
  "texto": "La estandarización resta la media y divide por la desviación estándar de cada variable; es importante en modelos sensibles a la escala, como los regularizados. Las variables categóricas se convierten en números con codificación one-hot, que crea una columna binaria por categoría y evita imponer un orden falso."},
 {"fuente_id": "apuntes_u2_a", "titulo": "Apuntes U2: regresión lineal", "url": "https://scikit-learn.org/stable/modules/linear_model.html",
  "texto": "La regresión lineal modela la variable respuesta como una combinación lineal de las variables de entrada más un intercepto. Los coeficientes se estiman por mínimos cuadrados, que minimizan la suma de los residuos al cuadrado. Cada coeficiente indica el cambio esperado en la respuesta por una unidad de cambio en esa variable, manteniendo las demás constantes."},
 {"fuente_id": "apuntes_u2_b", "titulo": "Apuntes U2: métricas de regresión", "url": "https://scikit-learn.org/stable/modules/model_evaluation.html",
  "texto": "El error cuadrático medio (MSE) promedia los errores al cuadrado y penaliza más los errores grandes. El coeficiente R² mide la proporción de la varianza de la respuesta explicada por el modelo; vale 1 en un ajuste perfecto y puede ser negativo si el modelo es peor que predecir la media."},
 {"fuente_id": "apuntes_u3_a", "titulo": "Apuntes U3: regresión logística y sigmoide", "url": "https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression",
  "texto": "La regresión logística es un modelo de clasificación binaria. Calcula una combinación lineal de las variables y la transforma con la función sigmoide, 1/(1+e^(-z)), que devuelve valores entre 0 y 1 interpretables como la probabilidad de la clase positiva. Para decidir la clase se compara esa probabilidad con un umbral, por defecto 0,5, que puede ajustarse según el costo de los errores."},
 {"fuente_id": "apuntes_u3_b", "titulo": "Apuntes U3: log-loss, log-odds y regularización", "url": "https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression",
  "texto": "Los parámetros de la regresión logística se estiman minimizando la pérdida logarítmica (log-loss o entropía cruzada), que castiga con fuerza las predicciones seguras y equivocadas. Cada coeficiente es el cambio en el logaritmo de las odds de la clase positiva por unidad de la variable; exp(coeficiente) es la razón de odds. La regularización L2 agrega una penalización a los coeficientes grandes para reducir el sobreajuste; en scikit-learn su fuerza se controla con C, donde un C menor implica más regularización."},
 {"fuente_id": "apuntes_u4_a", "titulo": "Apuntes U4: matriz de confusión y métricas", "url": "https://scikit-learn.org/stable/modules/model_evaluation.html#classification-metrics",
  "texto": "La matriz de confusión cuenta verdaderos positivos, falsos positivos, verdaderos negativos y falsos negativos. La precisión es VP/(VP+FP): de lo que el modelo marcó positivo, cuánto lo era. El recall es VP/(VP+FN): de los positivos reales, cuántos detectó. F1 es la media armónica de precisión y recall. Con clases desbalanceadas la exactitud puede ser engañosa."},
 {"fuente_id": "apuntes_u4_b", "titulo": "Apuntes U4: ROC, AUC y validación cruzada", "url": "https://scikit-learn.org/stable/modules/cross_validation.html",
  "texto": "La curva ROC grafica la tasa de verdaderos positivos contra la de falsos positivos al variar el umbral; el AUC resume la curva y equivale a la probabilidad de que el modelo puntúe más alto a un positivo que a un negativo elegidos al azar. La validación cruzada k-fold divide el entrenamiento en k partes, entrena k veces dejando una parte para validar y promedia los resultados, lo que da una estimación más estable que una sola partición."},
 {"fuente_id": "apuntes_u5_a", "titulo": "Apuntes U5: árboles de decisión", "url": "https://scikit-learn.org/stable/modules/tree.html",
  "texto": "Un árbol de decisión divide los datos con preguntas sobre las variables, eligiendo en cada nodo la división que más reduce la impureza, medida con el índice de Gini o la entropía. Los árboles profundos memorizan el entrenamiento y sobreajustan; limitar la profundidad o el número mínimo de muestras por hoja lo controla."},
 {"fuente_id": "apuntes_u5_b", "titulo": "Apuntes U5: bosques aleatorios", "url": "https://scikit-learn.org/stable/modules/ensemble.html#forests-of-randomized-trees",
  "texto": "Un bosque aleatorio entrena muchos árboles sobre muestras bootstrap y con subconjuntos aleatorios de variables en cada división, y promedia sus predicciones; esto reduce la varianza frente a un solo árbol. La importancia de variables por reducción de impureza indica cuánto contribuye cada variable a las divisiones, aunque puede favorecer variables con muchas categorías."},
]

# Fuentes propias en fuentes/*.txt|md → fragmentos de ~900 caracteres
for archivo in sorted(pathlib.Path("fuentes").glob("*")) if pathlib.Path("fuentes").exists() else []:
    if archivo.suffix.lower() in {".txt", ".md"}:
        texto = archivo.read_text(encoding="utf-8")
        partes = [p.strip() for p in re.split(r"\n\s*\n", texto) if p.strip()]
        buf, k = "", 0
        for parte in partes + [""]:
            if (len(buf) + len(parte) > 900 or not parte) and buf:
                k += 1
                FUENTES.append({"fuente_id": f"{archivo.stem}_{k}", "titulo": f"{archivo.stem} (fragmento {k})",
                                "url": archivo.name, "texto": buf.strip()})
                buf = ""
            buf += "\n" + parte
print(len(FUENTES), "fragmentos de fuentes")

# Estudiante ficticia (datos de ejemplo, sin datos personales reales)
ESTUDIANTE = {
    "usuario_id": "u_laura",
    "nombre": "Laura (estudiante ficticia)",
    "email": "laura.ejemplo@example.org",
    "inscripcion_id": "insc_laura_ml_ruta1",
    "perfil": ("Soy ingeniera industrial y trabajo hace dos años como analista de datos en una empresa de logística. "
               "Uso Excel y SQL todos los días y hago tableros en Power BI. En la universidad vi estadística y regresión "
               "lineal, y este año hice un curso corto de Python con pandas. Nunca he entrenado un modelo de clasificación "
               "y no tengo claro cómo se evalúa un modelo más allá del R². Me interesa predecir qué pedidos llegarán tarde."),
}
RUTA_ID = "ruta_ml_intro_v1"
USAR_OBJETIVO_DE_RUTA = False  # True = usar el objetivo por defecto de la ruta sin pedirle uno al LLM

## 6. Mago de Oz — Paso 1: syllabus estructurado (LLM) + validación del docente
El LLM extrae temas, resultados de aprendizaje, Bloom y prerrequisitos. El código verifica que sea un DAG (ids únicos, prerrequisitos existentes, sin ciclos) antes de que el docente lo apruebe.

In [ ]:
syllabus_llm, meta_syl = llm_json("syllabus_estructurado", SyllabusLLM, {"syllabus": SYLLABUS_TEXTO}, temperature=0.1)

def orden_topologico(temas: list[TemaSyllabus]) -> list[str]:
    ids = [t.id for t in temas]
    assert len(ids) == len(set(ids)), f"ids de tema repetidos: {ids}"
    faltan = {(t.id, p) for t in temas for p in t.prerrequisitos if p not in ids}
    assert not faltan, f"prerrequisitos inexistentes: {faltan}"
    pendientes = {t.id: set(t.prerrequisitos) - {t.id} for t in temas}
    orden = []
    while pendientes:
        listos = [i for i in ids if i in pendientes and not pendientes[i]]
        assert listos, f"Ciclo en prerrequisitos: {pendientes}"
        for i in listos:
            orden.append(i)
            del pendientes[i]
        for deps in pendientes.values():
            deps.difference_update(listos)
    return orden

ORDEN = orden_topologico(syllabus_llm.temas)
TEMAS = {t.id: t for t in syllabus_llm.temas}

display(pd.DataFrame([{
    "orden": ORDEN.index(t.id) + 1, "id": t.id, "nombre": t.nombre, "bloom": t.nivel_bloom,
    "prerrequisitos": ", ".join(t.prerrequisitos) or "—", "resultados": len(t.resultados_aprendizaje),
    "horas": t.horas_estimadas} for t in syllabus_llm.temas]).sort_values("orden"))
print("Objetivo por defecto:", syllabus_llm.objetivo_por_defecto)

**Revisión del docente.** Si algo está mal, corrígelo aquí (ej. `TEMAS["regresion_logistica"].prerrequisitos = [...]`), vuelve a correr la verificación y marca la aprobación. Sin esta validación humana todo lo demás hereda los errores.

In [ ]:
CORRECCIONES_DOCENTE = {}   # ej. {"regresion_logistica": {"horas_estimadas": 8}}
for tid, cambios in CORRECCIONES_DOCENTE.items():
    for campo, valor in cambios.items():
        setattr(TEMAS[tid], campo, valor)
ORDEN = orden_topologico(list(TEMAS.values()))

syllabus = Syllabus(**syllabus_llm.model_dump(), aprobado_por_docente=True, meta=meta_syl)
guardar("syllabus", syllabus)
print("✅ Syllabus aprobado:", len(syllabus.temas), "temas | prompt", meta_syl.prompt_version)

## 7. Mago de Oz — Paso 2: diagnóstico (LLM) + perfil semántico (embeddings)

1. Gemini genera 3 preguntas por tema (Recordar / Aplicar / Analizar).
2. El estudiante simulado responde (en el piloto, aquí van las respuestas reales).
3. Nivel por tema = aciertos ponderados por dificultad.
4. El perfil escrito se compara con cada tema mediante embeddings y se combina con la prueba (la prueba pesa más; si la persona escribió poco, el perfil pesa menos, para no penalizar a quien escribe menos).

In [ ]:
temas_json = json.dumps([{"id": t.id, "nombre": t.nombre, "subtemas": t.subtemas,
                          "resultados_aprendizaje": [r.enunciado for r in t.resultados_aprendizaje]}
                         for t in syllabus.temas], ensure_ascii=False)
diag_llm, meta_diag = llm_json("diagnostico", DiagnosticoLLM, {"temas": temas_json}, temperature=0.5)

# Validaciones semánticas que el esquema no puede expresar
problemas = []
for q in diag_llm.preguntas:
    if q.tema_id not in TEMAS: problemas.append(f"{q.id}: tema inexistente {q.tema_id}")
    if len(q.opciones) != 4: problemas.append(f"{q.id}: {len(q.opciones)} opciones")
    if not 0 <= q.respuesta_correcta < len(q.opciones): problemas.append(f"{q.id}: índice correcto fuera de rango")
    q.dificultad = min(3, max(1, q.dificultad))
cobertura = pd.Series([q.tema_id for q in diag_llm.preguntas]).value_counts()
print("Preguntas por tema:", cobertura.to_dict())
print("Problemas:", problemas or "ninguno")
diag_llm.preguntas = [q for q in diag_llm.preguntas if q.tema_id in TEMAS and 0 <= q.respuesta_correcta < len(q.opciones)]

def preguntas_para_estudiante(preguntas):
    """Lo que ve la pantalla del estudiante: sin respuesta correcta ni justificación."""
    return [{"id": q.id, "enunciado": q.enunciado, "tipo": getattr(q, "tipo", "opcion_multiple"),
             "opciones": q.opciones} for q in preguntas]

def simular_estudiante(preguntas, contexto: str):
    r, meta = llm_json("estudiante_simulado", RespuestasSimuladas, {
        "perfil": ESTUDIANTE["perfil"], "contexto": contexto or "Nada todavía (es el diagnóstico inicial).",
        "preguntas": json.dumps(preguntas_para_estudiante(preguntas), ensure_ascii=False)}, temperature=0.7)
    por_id = {x.pregunta_id: x for x in r.respuestas}
    # Toda pregunta queda con respuesta (si el simulador omitió alguna, cuenta como no respondida)
    respuestas = [por_id.get(q.id, RespuestaEstudiante(pregunta_id=q.id, opcion_elegida=-1, texto=""))
                  for q in preguntas]
    return respuestas, meta

# --- Respuestas: estudiante simulado (reemplazar por las reales del piloto) ---
resp_diag, meta_resp_diag = simular_estudiante(diag_llm.preguntas, "")

aciertos = {q.id: next(r.opcion_elegida for r in resp_diag if r.pregunta_id == q.id) == q.respuesta_correcta
            for q in diag_llm.preguntas}
puntaje_prueba = {}
for tid in TEMAS:
    qs = [q for q in diag_llm.preguntas if q.tema_id == tid]
    peso = sum(q.dificultad for q in qs)
    puntaje_prueba[tid] = sum(q.dificultad * aciertos[q.id] for q in qs) / peso if peso else 0.0

# --- Perfil semántico con embeddings ---
textos_tema = [f"{t.nombre}. {t.descripcion} Subtemas: {', '.join(t.subtemas)}." for t in syllabus.temas]
E_temas = embed(textos_tema, "SEMANTIC_SIMILARITY")
E_perfil = embed([ESTUDIANTE["perfil"]], "SEMANTIC_SIMILARITY")[0]
sims = E_temas @ E_perfil
rango = sims.max() - sims.min()
priors = (sims - sims.min()) / rango if rango > 1e-6 else np.zeros_like(sims)
palabras = len(ESTUDIANTE["perfil"].split())
PESO_PERFIL = 0.25 * min(1.0, palabras / 60)   # heurística a calibrar con datos del piloto

niveles = []
for i, t in enumerate(syllabus.temas):
    combinado = (1 - PESO_PERFIL) * puntaje_prueba[t.id] + PESO_PERFIL * float(priors[i])
    niveles.append(NivelTema(tema_id=t.id, puntaje_prueba=round(puntaje_prueba[t.id], 3),
                             similitud_perfil=round(float(sims[i]), 4), prior_perfil=round(float(priors[i]), 3),
                             peso_perfil=round(PESO_PERFIL, 3), nivel_combinado=round(combinado, 3),
                             dominado=combinado >= UMBRAL_DOMINIO))

peso_total = sum(q.dificultad for q in diag_llm.preguntas)
diagnostico = Diagnostico(
    inscripcion_id=ESTUDIANTE["inscripcion_id"], preguntas=diag_llm.preguntas, respuestas=resp_diag,
    nivel_por_tema=niveles,
    puntaje_global=round(sum(q.dificultad * aciertos[q.id] for q in diag_llm.preguntas) / peso_total, 3),
    meta_preguntas=meta_diag, meta_respuestas=meta_resp_diag)
guardar("diagnostico", diagnostico)
display(pd.DataFrame([n.model_dump() for n in niveles]))
print("Ejemplo de pregunta:"); ver(diag_llm.preguntas[0], 800)

## 8. Mago de Oz — Paso 3: objetivo de aprendizaje (LLM)

In [ ]:
brecha = {n.tema_id: n.nivel_combinado for n in niveles}
if USAR_OBJETIVO_DE_RUTA:
    objetivo = Objetivo(inscripcion_id=ESTUDIANTE["inscripcion_id"], origen="ruta", confirmado_por_estudiante=True,
                        enunciado=syllabus.objetivo_por_defecto, verbo_bloom="", contenido="", criterio="",
                        temas_foco=[n.tema_id for n in niveles if not n.dominado], justificacion="Objetivo por defecto de la ruta")
else:
    obj_llm, meta_obj = llm_json("objetivo_aprendizaje", ObjetivoLLM, {
        "objetivo_ruta": syllabus.objetivo_por_defecto, "perfil": ESTUDIANTE["perfil"],
        "brecha": json.dumps(brecha, ensure_ascii=False)})
    obj_llm.temas_foco = [t for t in obj_llm.temas_foco if t in TEMAS]
    objetivo = Objetivo(**obj_llm.model_dump(), inscripcion_id=ESTUDIANTE["inscripcion_id"], origen="propuesto_llm",
                        confirmado_por_estudiante=True,  # en la app, el estudiante lo confirma o ajusta
                        meta=meta_obj)
guardar("objetivo", objetivo)
print("🎯", objetivo.enunciado)
print("Temas foco:", objetivo.temas_foco)

## 9. Secuenciación de la ruta (determinista, sin LLM)
Orden topológico del DAG; se omiten los temas con dominio ≥ umbral y se incluyen los prerrequisitos débiles de cada tema incluido.

In [ ]:
def secuenciar(niveles: list[NivelTema]) -> RutaPersonalizada:
    nivel = {n.tema_id: n.nivel_combinado for n in niveles}
    incluir = {tid for tid in ORDEN if nivel[tid] < UMBRAL_DOMINIO}
    cambio = True
    while cambio:  # cierre: prerrequisitos débiles de los temas incluidos
        cambio = False
        for tid in list(incluir):
            for p in TEMAS[tid].prerrequisitos:
                if p not in incluir and nivel[p] < UMBRAL_DOMINIO:
                    incluir.add(p); cambio = True
    pasos = []
    for k, tid in enumerate(ORDEN, start=1):
        inc = tid in incluir
        pasos.append(PasoRuta(
            orden=k, tema_id=tid, nombre=TEMAS[tid].nombre, incluido=inc,
            motivo=(f"nivel {nivel[tid]:.0%} < {UMBRAL_DOMINIO:.0%}" if inc else f"dominado en diagnóstico ({nivel[tid]:.0%})"),
            estado="pendiente" if inc else "omitido", puntaje=None if inc else round(nivel[tid], 3)))
    return RutaPersonalizada(inscripcion_id=ESTUDIANTE["inscripcion_id"], ruta_id=RUTA_ID,
                             algoritmo="topologico_umbral v1", umbral_dominio=UMBRAL_DOMINIO, pasos=pasos)

ruta = secuenciar(niveles)
display(pd.DataFrame([p.model_dump() for p in ruta.pasos]))

## 10. Mago de Oz — Paso 4: un tema completo (RAG + video + quiz)

- **Texto**: se recuperan los fragmentos más cercanos al tema por embeddings (`RETRIEVAL_DOCUMENT` / `RETRIEVAL_QUERY`) y Gemini redacta citando `[n]`. Un segundo prompt verifica la fidelidad (métrica tipo RAGAS).
- **Video**: con `YOUTUBE_API_KEY` se buscan candidatos; si no, se usa la lista curada del docente. En ambos casos se re-ordenan por similitud de embeddings con el tema y se filtran por duración.
- **Quiz**: se genera desde los resultados de aprendizaje.

In [ ]:
E_fuentes = embed([f"{f['titulo']}. {f['texto']}" for f in FUENTES], "RETRIEVAL_DOCUMENT")

def recuperar(tema: TemaSyllabus, k: int = 4):
    consulta = f"{tema.nombre}: {', '.join(tema.subtemas)}. " + " ".join(r.enunciado for r in tema.resultados_aprendizaje)
    q = embed([consulta], "RETRIEVAL_QUERY")[0]
    s = E_fuentes @ q
    top = np.argsort(-s)[:k]
    return [(n + 1, FUENTES[i], float(s[i])) for n, i in enumerate(top)]

def fragmentos_texto(recs):
    return "\n\n".join(f"[{n}] ({f['titulo']}) {f['texto']}" for n, f, _ in recs)

def tema_json(tema):
    return json.dumps({"id": tema.id, "nombre": tema.nombre, "subtemas": tema.subtemas,
                       "resultados_aprendizaje": [r.model_dump() for r in tema.resultados_aprendizaje]},
                      ensure_ascii=False)

def texto_explicacion(e) -> str:
    return "\n".join([e.titulo] + [f"{s.subtitulo}: {s.contenido}" for s in e.secciones] + [e.ejemplo, e.resumen])

def generar_explicacion(tid: str, tipo="principal", errores: list[str] | None = None, verificar=False):
    tema = TEMAS[tid]
    recs = recuperar(tema)
    nivel = next(n.nivel_combinado for n in niveles if n.tema_id == tid)
    modo = ("Esta es la explicación principal del tema." if tipo == "principal" else
            "El estudiante NO aprobó el quiz. Escribe una explicación ALTERNA con otro enfoque "
            "(analogías y paso a paso), centrada en estos errores: " + json.dumps(errores or [], ensure_ascii=False))
    e, meta = llm_json("explicacion_tema", ExplicacionLLM, {
        "tema": tema_json(tema), "nivel": round(nivel, 2), "perfil": ESTUDIANTE["perfil"],
        "modo": modo, "fragmentos": fragmentos_texto(recs)})
    citas = {int(c) for c in re.findall(r"\[(\d+)\]", texto_explicacion(e))}
    citas_invalidas = sorted(c for c in citas if c < 1 or c > len(recs))
    secciones_sin_cita = [s.subtitulo for s in e.secciones if not re.search(r"\[\d+\]", s.contenido)]
    fid = None
    if verificar:
        v, _ = llm_json("fidelidad_rag", VerificacionFidelidad, {
            "texto": texto_explicacion(e), "fragmentos": fragmentos_texto(recs)}, temperature=0.0)
        fid = round(sum(a.soportada for a in v.afirmaciones) / max(1, len(v.afirmaciones)), 3)
        no_soportadas = [a.texto for a in v.afirmaciones if not a.soportada]
        if no_soportadas: print("  Afirmaciones no soportadas:", no_soportadas)
    exp = ExplicacionTema(**e.model_dump(), tema_id=tid, tipo=tipo, fidelidad=fid, meta=meta,
                          fuentes=[FuenteCitada(n=n, fuente_id=f["fuente_id"], titulo=f["titulo"], url=f["url"])
                                   for n, f, _ in recs])
    print(f"  {tid} [{tipo}] citas={sorted(citas)} inválidas={citas_invalidas} "
          f"secciones sin cita={secciones_sin_cita} fidelidad={fid}")
    return exp, recs

# Tema demo: regresión logística si existe; si no, el primero de la ruta
TEMA_DEMO = next((tid for tid in ORDEN if "logist" in tid),
                 next((p.tema_id for p in ruta.pasos if p.incluido), ORDEN[0]))
explicacion_demo, recs_demo = generar_explicacion(TEMA_DEMO, verificar=True)
guardar(f"explicacion_{TEMA_DEMO}", explicacion_demo)

print("\n#", explicacion_demo.titulo)
for s in explicacion_demo.secciones:
    print(f"\n## {s.subtitulo}\n{textwrap.fill(s.contenido, 110)}")
print("\nEjemplo:", textwrap.fill(explicacion_demo.ejemplo, 110))
print("\nFuentes:", [(f.n, f.titulo) for f in explicacion_demo.fuentes])

In [ ]:
import requests

YOUTUBE_API_KEY = os.environ.get("YOUTUBE_API_KEY")
if not YOUTUBE_API_KEY:
    try:
        from google.colab import userdata
        YOUTUBE_API_KEY = userdata.get("YOUTUBE_API_KEY")
    except Exception:
        pass

# Lista curada del docente (respaldo). Formato: {"video_id": "...", "titulo": "...", "canal": "...", "descripcion": "...", "duracion_min": 12}
VIDEOS_DOCENTE = {
    # "regresion_logistica": [ {...}, ... ],
}
DURACION_MIN, DURACION_MAX = 4, 30

def _iso8601_min(d):
    m = re.match(r"PT(?:(\d+)H)?(?:(\d+)M)?(?:(\d+)S)?", d or "")
    return None if not m else int(m[1] or 0) * 60 + int(m[2] or 0) + int(m[3] or 0) / 60

def buscar_youtube(consulta, n=10):
    r = requests.get("https://www.googleapis.com/youtube/v3/search", timeout=20, params={
        "key": YOUTUBE_API_KEY, "part": "snippet", "q": consulta, "type": "video",
        "maxResults": n, "relevanceLanguage": "es", "videoEmbeddable": "true"})
    r.raise_for_status()
    items = r.json().get("items", [])
    ids = [it["id"]["videoId"] for it in items]
    dur = {}
    if ids:
        d = requests.get("https://www.googleapis.com/youtube/v3/videos", timeout=20, params={
            "key": YOUTUBE_API_KEY, "part": "contentDetails", "id": ",".join(ids)}).json()
        dur = {v["id"]: _iso8601_min(v["contentDetails"]["duration"]) for v in d.get("items", [])}
    return [{"video_id": it["id"]["videoId"], "titulo": it["snippet"]["title"], "canal": it["snippet"]["channelTitle"],
             "descripcion": it["snippet"]["description"], "duracion_min": dur.get(it["id"]["videoId"])} for it in items]

def recurso_video(tid: str) -> RecursoVideo:
    tema = TEMAS[tid]
    if YOUTUBE_API_KEY:
        origen, cands = "youtube_api", buscar_youtube(f"{tema.nombre} machine learning explicación")
    elif VIDEOS_DOCENTE.get(tid):
        origen, cands = "lista_docente", VIDEOS_DOCENTE[tid]
    else:
        print("  Sin YOUTUBE_API_KEY ni lista del docente: el tema queda sin video.")
        return RecursoVideo(tema_id=tid, origen="ninguno", candidatos=[], seleccionado=None, embed_model=EMBED_MODEL)
    cands = [c for c in cands if c.get("duracion_min") is None or DURACION_MIN <= c["duracion_min"] <= DURACION_MAX]
    if not cands:
        return RecursoVideo(tema_id=tid, origen=origen, candidatos=[], seleccionado=None, embed_model=EMBED_MODEL)
    # Re-ranking semántico: título + descripción (o transcripción, si se agrega) vs. tema
    E_c = embed([f"{c['titulo']}. {c.get('descripcion', '')}" for c in cands], "SEMANTIC_SIMILARITY")
    E_t = embed([f"{tema.nombre}. {tema.descripcion}"], "SEMANTIC_SIMILARITY")[0]
    sims_v = E_c @ E_t
    ordenados = sorted(zip(cands, sims_v), key=lambda x: -x[1])
    candidatos = [VideoCandidato(video_id=c["video_id"], titulo=c["titulo"], canal=c.get("canal", ""),
                                 url=f"https://www.youtube.com/watch?v={c['video_id']}",
                                 duracion_min=c.get("duracion_min"), similitud=round(float(s), 4)) for c, s in ordenados]
    return RecursoVideo(tema_id=tid, origen=origen, candidatos=candidatos[:5], seleccionado=candidatos[0], embed_model=EMBED_MODEL)

video_demo = recurso_video(TEMA_DEMO)
guardar(f"video_{TEMA_DEMO}", video_demo)
display(pd.DataFrame([c.model_dump() for c in video_demo.candidatos]) if video_demo.candidatos else "sin candidatos")

In [ ]:
N_OM, N_RC = 4, 1

def generar_quiz(tid: str, intento: int, previas: list[str], recs) -> Quiz:
    q, meta = llm_json("quiz_tema", QuizLLM, {
        "intento": intento, "n_om": N_OM, "n_rc": N_RC, "tema_id": tid, "tema": tema_json(TEMAS[tid]),
        "previas": json.dumps(previas, ensure_ascii=False) if previas else "ninguna",
        "fragmentos": fragmentos_texto(recs)}, temperature=0.6)
    validas = []
    for p in q.preguntas:
        p.tema_id = tid
        if p.tipo == "opcion_multiple" and not (len(p.opciones) >= 2 and 0 <= p.respuesta_correcta < len(p.opciones)):
            print("  descartada (índice inválido):", p.id); continue
        if p.tipo == "respuesta_corta" and not p.rubrica:
            print("  descartada (sin rúbrica):", p.id); continue
        validas.append(p)
    return Quiz(preguntas=validas, quiz_id=f"quiz_{tid}_i{intento}", tema_id=tid, intento=intento,
                umbral_aprobacion=UMBRAL_TEMA, meta=meta)

quiz_demo = generar_quiz(TEMA_DEMO, 1, [], recs_demo)
guardar(f"quiz_{TEMA_DEMO}_i1", quiz_demo)
ver(quiz_demo, 2500)

## 11. Calificación (regla para opción múltiple, LLM-as-judge para respuesta corta)

In [ ]:
def calificar(evaluacion, respuestas, meta_resp: Meta, tipo="tema") -> Calificacion:
    por_id = {r.pregunta_id: r for r in respuestas}
    resultados, metas = [], []
    for p in evaluacion.preguntas:
        r = por_id[p.id]
        if p.tipo == "opcion_multiple":
            ok = r.opcion_elegida == p.respuesta_correcta
            resultados.append(ResultadoPregunta(pregunta_id=p.id, tema_id=p.tema_id, tipo=p.tipo, correcta=ok,
                                                puntaje=float(ok), calificador="regla",
                                                retroalimentacion="Correcto." if ok else p.respuesta_modelo))
        else:
            total = sum(c.puntos for c in p.rubrica) or 1.0
            if not r.texto.strip():
                resultados.append(ResultadoPregunta(pregunta_id=p.id, tema_id=p.tema_id, tipo=p.tipo, puntaje=0.0,
                                                    calificador="regla", retroalimentacion="Sin respuesta."))
                continue
            c, meta = llm_json("calificacion_respuesta_corta", CalificacionLLM, {
                "pregunta": p.enunciado, "respuesta_modelo": p.respuesta_modelo,
                "rubrica": json.dumps([x.model_dump() for x in p.rubrica], ensure_ascii=False),
                "respuesta": r.texto}, temperature=0.0)
            maximos = {x.criterio: x.puntos for x in p.rubrica}
            otorgado = sum(min(max(0.0, k.puntos_otorgados), maximos.get(k.criterio, k.puntos_otorgados)) for k in c.criterios)
            puntaje = round(min(1.0, otorgado / total), 3)
            metas.append(meta)
            resultados.append(ResultadoPregunta(pregunta_id=p.id, tema_id=p.tema_id, tipo=p.tipo, correcta=puntaje >= 0.6,
                                                puntaje=puntaje, calificador="llm_juez",
                                                retroalimentacion=c.retroalimentacion, detalle_rubrica=c.criterios))
    total = round(sum(x.puntaje for x in resultados) / max(1, len(resultados)), 3)
    eid = evaluacion.quiz_id if tipo == "tema" else evaluacion.evaluacion_id
    return Calificacion(intento_id=f"{eid}_{ESTUDIANTE['usuario_id']}", evaluacion_id=eid, tipo=tipo,
                        tema_id=getattr(evaluacion, "tema_id", None), respuestas=respuestas, por_pregunta=resultados,
                        puntaje_total=total, aprobado=total >= evaluacion.umbral_aprobacion, fecha=_ahora(),
                        meta_respuestas=meta_resp, meta_calificador=metas)

def contexto_estudiado(explicaciones: list[ExplicacionTema], max_chars=3500):
    return "\n\n".join(texto_explicacion(e) for e in explicaciones)[-max_chars:]

resp_demo, meta_resp_demo = simular_estudiante(quiz_demo.preguntas, contexto_estudiado([explicacion_demo]))
cal_demo = calificar(quiz_demo, resp_demo, meta_resp_demo)
display(pd.DataFrame([{k: v for k, v in r.model_dump().items() if k != "detalle_rubrica"} for r in cal_demo.por_pregunta]))
print(f"Puntaje: {cal_demo.puntaje_total:.0%} → {'aprobado' if cal_demo.aprobado else 'no aprobado'}")

## 12. Recorrido completo de la ruta (bucle por tema + evaluación final + badge)

Ejecuta el flujo del brief para Laura: por cada tema incluido, explicación → quiz → calificación; si no aprueba, explicación alterna y quiz nuevo (máx. 3). Todo se registra en un único estado, del que salen todas las pantallas. Las salidas del tema demo se reutilizan de la caché (mismo prompt, misma entrada → mismo contenido, como en producción).

In [ ]:
EVENTOS, CONTENIDO = [], {}
ruta_viva = ruta.model_copy(deep=True)
SNAPSHOTS = {}

def evento(tipo, **detalle):
    EVENTOS.append({"t": _ahora(), "tipo": tipo, **detalle})

evento("inscripcion", ruta_id=RUTA_ID)
evento("diagnostico_completado", puntaje_global=diagnostico.puntaje_global)
evento("objetivo_confirmado", objetivo=objetivo.enunciado)

estudiado = []
for paso in ruta_viva.pasos:
    if not paso.incluido:
        evento("tema_omitido", tema_id=paso.tema_id, nivel=paso.puntaje)
        continue
    tid = paso.tema_id
    paso.estado = "en_curso"
    exp, recs = generar_explicacion(tid, verificar=(tid == TEMA_DEMO))
    CONTENIDO[tid] = {"explicaciones": [exp], "video": video_demo if tid == TEMA_DEMO else None,
                      "quizzes": [], "calificaciones": []}
    previas = []
    for intento in range(1, MAX_INTENTOS + 1):
        quiz = generar_quiz(tid, intento, previas, recs)
        previas += [p.enunciado for p in quiz.preguntas]
        resp, meta_r = simular_estudiante(quiz.preguntas, contexto_estudiado(estudiado + CONTENIDO[tid]["explicaciones"]))
        cal = calificar(quiz, resp, meta_r)
        CONTENIDO[tid]["quizzes"].append(quiz); CONTENIDO[tid]["calificaciones"].append(cal)
        paso.intentos, paso.puntaje = intento, cal.puntaje_total
        evento("quiz_calificado", tema_id=tid, intento=intento, puntaje=cal.puntaje_total, aprobado=cal.aprobado)
        print(f"{tid}: intento {intento} → {cal.puntaje_total:.0%}")
        if cal.aprobado:
            paso.estado = "aprobado"
            break
        errores = [next(p.enunciado for p in quiz.preguntas if p.id == r.pregunta_id)
                   for r in cal.por_pregunta if r.puntaje < 0.6]
        if intento < MAX_INTENTOS:
            ref, _ = generar_explicacion(tid, tipo="refuerzo", errores=errores)
            CONTENIDO[tid]["explicaciones"].append(ref)
            evento("refuerzo_entregado", tema_id=tid, intento=intento)
    else:
        paso.estado = "bloqueado"
        evento("tema_bloqueado", tema_id=tid)
    estudiado += CONTENIDO[tid]["explicaciones"]
    if tid == TEMA_DEMO:
        SNAPSHOTS["en_curso"] = {"ruta": ruta_viva.model_copy(deep=True), "n_eventos": len(EVENTOS)}
    if paso.estado == "bloqueado":
        print("⛔ Tema bloqueado tras", MAX_INTENTOS, "intentos: requiere intervención del docente.")
        break

In [ ]:
# Evaluación final → badge
evaluacion_final, cal_final, BADGE = None, None, None
temas_ruta = [p.tema_id for p in ruta_viva.pasos if p.incluido]
ruta_completa = all(p.estado in ("aprobado", "omitido") for p in ruta_viva.pasos)

if ruta_completa:
    temas_final_json = json.dumps([json.loads(tema_json(TEMAS[t])) for t in temas_ruta], ensure_ascii=False)
    previas_final = []
    for intento in range(1, MAX_INTENTOS + 1):
        ef, meta_ef = llm_json("evaluacion_final", QuizLLM, {
            "intento": intento, "n_rc": 2, "temas": temas_final_json,
            "previas": json.dumps(previas_final, ensure_ascii=False) if previas_final else "ninguna"}, temperature=0.6)
        ef.preguntas = [p for p in ef.preguntas if p.tema_id in TEMAS and (
            (p.tipo == "opcion_multiple" and 0 <= p.respuesta_correcta < len(p.opciones)) or
            (p.tipo == "respuesta_corta" and p.rubrica))]
        evaluacion_final = EvaluacionFinal(preguntas=ef.preguntas, evaluacion_id=f"final_{RUTA_ID}_i{intento}",
                                           temas_evaluados=temas_ruta, intento=intento,
                                           umbral_aprobacion=UMBRAL_FINAL, meta=meta_ef)
        previas_final += [p.enunciado for p in ef.preguntas]
        resp_f, meta_rf = simular_estudiante(evaluacion_final.preguntas, contexto_estudiado(estudiado, 6000))
        cal_final = calificar(evaluacion_final, resp_f, meta_rf, tipo="final")
        evento("evaluacion_final", intento=intento, puntaje=cal_final.puntaje_total, aprobado=cal_final.aprobado)
        print(f"Evaluación final intento {intento}: {cal_final.puntaje_total:.0%}")
        if cal_final.aprobado:
            break
        por_tema = pd.DataFrame([r.model_dump() for r in cal_final.por_pregunta]).groupby("tema_id")["puntaje"].mean()
        debiles = [t for t, v in por_tema.items() if v < UMBRAL_TEMA]
        evento("repaso_dirigido", temas=debiles)
        for t in debiles:
            ref, _ = generar_explicacion(t, tipo="refuerzo", errores=[f"Bajo desempeño en evaluación final ({por_tema[t]:.0%})"])
            CONTENIDO[t]["explicaciones"].append(ref); estudiado.append(ref)
    guardar("evaluacion_final", evaluacion_final)
    guardar("calificacion_final", cal_final)
else:
    print("La ruta no está completa; no se habilita la evaluación final.")

def svg_badge(nombre: str) -> str:
    lineas = textwrap.wrap(nombre, 18)[:3]
    texto = "".join(f'<text x="120" y="{112 + i * 22}" text-anchor="middle" font-family="Arial" font-size="17" '
                    f'font-weight="bold" fill="#ffffff">{l}</text>' for i, l in enumerate(lineas))
    return (f'<svg xmlns="http://www.w3.org/2000/svg" width="240" height="240" viewBox="0 0 240 240">'
            f'<circle cx="120" cy="120" r="112" fill="#1f4e79"/><circle cx="120" cy="120" r="98" fill="none" '
            f'stroke="#f2c14e" stroke-width="5"/><text x="120" y="78" text-anchor="middle" font-family="Arial" '
            f'font-size="13" fill="#f2c14e">SMARTMENTOR</text>{texto}</svg>')

if cal_final and cal_final.aprobado:
    bt, meta_bt = llm_json("badge_texto", BadgeTextoLLM, {
        "curso": syllabus.curso_nombre, "objetivo": objetivo.enunciado,
        "temas": ", ".join(TEMAS[t].nombre for t in temas_ruta), "umbral": f"{UMBRAL_FINAL:.0%}"})
    (OUT / "salidas" / "badge.svg").write_text(svg_badge(bt.nombre), encoding="utf-8")
    sal = hashlib.sha256(ESTUDIANTE["inscripcion_id"].encode()).hexdigest()[:12]
    BASE = "https://example.org/smartmentor"   # reemplazar por el dominio real de la plataforma
    BADGE = BadgeAssertion(
        id=f"{BASE}/assertions/{ESTUDIANTE['inscripcion_id']}",
        recipient=OBRecipient(salt=sal, identity="sha256$" + hashlib.sha256((ESTUDIANTE["email"] + sal).encode()).hexdigest()),
        badge=OBBadgeClass(id=f"{BASE}/badges/{RUTA_ID}", name=bt.nombre, description=bt.descripcion,
                           image=f"{BASE}/badges/{RUTA_ID}/badge.svg", criteria=OBCriteria(narrative=bt.criterios),
                           issuer=OBIssuer(id=f"{BASE}/issuer", name="SmartMentor (prototipo)", url=BASE),
                           tags=bt.competencias),
        issuedOn=_ahora(),
        evidence=[OBEvidence(narrative=f"Evaluación final {cal_final.puntaje_total:.0%} (umbral {UMBRAL_FINAL:.0%}); "
                                       f"prompt {meta_bt.prompt_version}")])
    guardar("badge", BADGE)
    evento("badge_emitido", badge_id=BADGE.id)
    ver(BADGE)
    from IPython.display import SVG
    display(SVG(svg_badge(bt.nombre)))
else:
    print("Sin badge: la evaluación final no alcanzó el umbral (o no se habilitó).")

SNAPSHOTS["final"] = {"ruta": ruta_viva.model_copy(deep=True), "n_eventos": len(EVENTOS)}
_ = SNAPSHOTS.setdefault("en_curso", SNAPSHOTS["final"])

## 13. Fuente única de datos → pantallas

Todas las pantallas se derivan de **un solo JSON**. Así se elimina la incoherencia que señaló la revisión (6 vs 5 temas, tres puntajes distintos para el mismo tema, badge antes de terminar la ruta).

Se exportan dos escenarios internamente coherentes: `en_curso` (justo después del tema demo) y `final`. Ajusta los ids de pantalla a los de tu prototipo.

In [ ]:
FUENTE_UNICA = {
    "version_datos": "ejemplo_laura v1",
    "modelos": {"llm": LLM_MODEL, "embeddings": EMBED_MODEL},
    "curso": {"id": syllabus.curso_id, "nombre": syllabus.curso_nombre},
    "ruta": {"id": RUTA_ID, "objetivo_por_defecto": syllabus.objetivo_por_defecto,
             "umbrales": {"tema": UMBRAL_TEMA, "final": UMBRAL_FINAL, "dominio": UMBRAL_DOMINIO, "max_intentos": MAX_INTENTOS}},
    "estudiante": {k: v for k, v in ESTUDIANTE.items() if k != "email"},
    "syllabus": syllabus.model_dump(),
    "diagnostico": diagnostico.model_dump(),
    "objetivo": objetivo.model_dump(),
    "contenido": {tid: {"explicaciones": [e.model_dump() for e in c["explicaciones"]],
                        "video": c["video"].model_dump() if c["video"] else None,
                        "quizzes": [q.model_dump() for q in c["quizzes"]],
                        "calificaciones": [x.model_dump() for x in c["calificaciones"]]} for tid, c in CONTENIDO.items()},
    "evaluacion_final": evaluacion_final.model_dump() if evaluacion_final else None,
    "calificacion_final": cal_final.model_dump() if cal_final else None,
    "badge": BADGE.model_dump(by_alias=True) if BADGE else None,
    "eventos": EVENTOS,
    "snapshots": {k: {"ruta": v["ruta"].model_dump(), "n_eventos": v["n_eventos"]} for k, v in SNAPSHOTS.items()},
}
guardar("fuente_unica_laura", FUENTE_UNICA)

def pantallas(F: dict, escenario: str) -> dict:
    ruta_s = F["snapshots"][escenario]["ruta"]
    eventos_s = F["eventos"][:F["snapshots"][escenario]["n_eventos"]]
    n_temas = len(F["syllabus"]["temas"])
    pasos = ruta_s["pasos"]
    hechos = [p for p in pasos if p["estado"] in ("aprobado", "omitido")]
    actual = next((p for p in pasos if p["estado"] in ("en_curso", "pendiente", "bloqueado")), None)
    badge_ok = any(e["tipo"] == "badge_emitido" for e in eventos_s)
    tema_demo = F["contenido"].get(TEMA_DEMO)
    return {
        "catalogo": {"ruta_id": F["ruta"]["id"], "curso": F["curso"]["nombre"], "num_temas": n_temas,
                     "objetivo": F["ruta"]["objetivo_por_defecto"]},
        "D2_syllabus_docente": [{"id": t["id"], "nombre": t["nombre"], "bloom": t["nivel_bloom"],
                                 "prerrequisitos": t["prerrequisitos"],
                                 "resultados": [r["enunciado"] for r in t["resultados_aprendizaje"]]}
                                for t in F["syllabus"]["temas"]],
        "E3_diagnostico_preguntas": preguntas_para_estudiante([PreguntaOM(**q) for q in F["diagnostico"]["preguntas"]]),
        "E4_diagnostico_resultado": [{"tema_id": n["tema_id"], "nivel": n["nivel_combinado"], "dominado": n["dominado"]}
                                     for n in F["diagnostico"]["nivel_por_tema"]],
        "E5_vista_ruta": {"objetivo": F["objetivo"]["enunciado"], "total_temas": n_temas,
                          "completados": len(hechos),
                          "tema_actual": ({"tema_id": actual["tema_id"], "posicion": actual["orden"],
                                           "texto": f"Tema {actual['orden']} de {n_temas}"} if actual else None),
                          "pasos": [{"tema_id": p["tema_id"], "nombre": p["nombre"], "estado": p["estado"],
                                     "puntaje": p["puntaje"], "intentos": p["intentos"]} for p in pasos],
                          "evaluacion_final_habilitada": all(p["estado"] in ("aprobado", "omitido") for p in pasos)},
        "E6_tema": None if not tema_demo else {
            "tema_id": TEMA_DEMO, "explicacion": tema_demo["explicaciones"][0],
            "prompt_version_explicacion": tema_demo["explicaciones"][0]["meta"]["prompt_version"],
            "video": (tema_demo["video"] or {}).get("seleccionado"),
            "quiz": {"quiz_id": tema_demo["quizzes"][0]["quiz_id"],
                     "prompt_version": tema_demo["quizzes"][0]["meta"]["prompt_version"],
                     "preguntas": preguntas_para_estudiante([PreguntaQuiz(**q) for q in tema_demo["quizzes"][0]["preguntas"]])}},
        "D3_dashboard_docente": [{"estudiante": F["estudiante"]["nombre"], "progreso": f"{len(hechos)}/{n_temas}",
                                  **{p["tema_id"]: p["puntaje"] for p in pasos}}],
        "D4_detalle_estudiante": [{"tema_id": e["tema_id"], "intento": e["intento"], "puntaje": e["puntaje"]}
                                  for e in eventos_s if e["tipo"] == "quiz_calificado"],
        "E12_badge": F["badge"] if badge_ok else None,
    }

def verificar_coherencia(F: dict, P: dict, escenario: str) -> list[str]:
    errores = []
    n = len(F["syllabus"]["temas"])
    if P["catalogo"]["num_temas"] != n or len(P["D2_syllabus_docente"]) != n or len(P["E5_vista_ruta"]["pasos"]) != n:
        errores.append("número de temas distinto entre catálogo, syllabus y vista de ruta")
    ultimo = {}
    for fila in P["D4_detalle_estudiante"]:
        ultimo[fila["tema_id"]] = fila["puntaje"]
    for p in P["E5_vista_ruta"]["pasos"]:
        if p["estado"] == "omitido":
            continue
        if p["tema_id"] in ultimo and ultimo[p["tema_id"]] != p["puntaje"]:
            errores.append(f"{p['tema_id']}: E5={p['puntaje']} vs D4 último intento={ultimo[p['tema_id']]}")
        if P["D3_dashboard_docente"][0][p["tema_id"]] != p["puntaje"]:
            errores.append(f"{p['tema_id']}: E5 y D3 no coinciden")
    if P["E12_badge"] and not P["E5_vista_ruta"]["evaluacion_final_habilitada"]:
        errores.append("badge emitido sin terminar la ruta")
    if P["E12_badge"] and P["E5_vista_ruta"]["tema_actual"]:
        errores.append("badge emitido pero la vista de ruta muestra un tema pendiente")
    if P["E4_diagnostico_resultado"]:
        diag_ids = {x["tema_id"] for x in P["E4_diagnostico_resultado"]}
        if diag_ids != {t["id"] for t in F["syllabus"]["temas"]}:
            errores.append("el diagnóstico no cubre exactamente los temas del syllabus")
    return errores

for esc in ["en_curso", "final"]:
    P = pantallas(FUENTE_UNICA, esc)
    guardar(f"pantallas_{esc}", P)
    errs = verificar_coherencia(FUENTE_UNICA, P, esc)
    print(f"Escenario {esc}: {'✅ coherente' if not errs else '❌ ' + '; '.join(errs)}")
    print("   E5:", P["E5_vista_ruta"]["tema_actual"], "| completados", P["E5_vista_ruta"]["completados"],
          "| badge:", "sí" if P["E12_badge"] else "no")

## 14. Validación de contratos (lo mismo que correría la integración continua)
Cada salida guardada se valida contra su esquema JSON exportado.

In [ ]:
import jsonschema

def validar(nombre_esquema, data):
    esquema = json.loads((OUT / "schemas" / f"{nombre_esquema}.v1.json").read_text(encoding="utf-8"))
    jsonschema.validate(data, esquema)

chequeos = [("syllabus", syllabus), ("diagnostico", diagnostico), ("objetivo", objetivo),
            ("ruta", ruta_viva), ("explicacion_tema", explicacion_demo), ("video", video_demo),
            ("quiz", quiz_demo), ("calificacion", cal_demo)]
for c in CONTENIDO.values():
    chequeos += [("explicacion_tema", e) for e in c["explicaciones"]] + [("quiz", q) for q in c["quizzes"]] + \
                [("calificacion", x) for x in c["calificaciones"]]
if evaluacion_final: chequeos += [("evaluacion_final", evaluacion_final), ("calificacion", cal_final)]
if BADGE: chequeos.append(("badge", BADGE))

fallos = 0
for nombre, obj in chequeos:
    try:
        validar(nombre, json.loads(obj.model_dump_json(by_alias=True)))
    except jsonschema.ValidationError as e:
        fallos += 1; print("❌", nombre, e.message[:200])
print(f"{len(chequeos) - fallos}/{len(chequeos)} salidas cumplen su esquema")

## 15. Métricas de la prueba (log del LLM + primeras métricas del brief)
Es una prueba de factibilidad con una estudiante simulada, no un resultado del piloto; sirve para ver si las salidas son útiles y cuánto cuestan antes de construir el pipeline.

In [ ]:
log = pd.read_json(LOG_PATH, lines=True)
resumen = log.groupby("tarea").agg(
    llamadas=("tarea", "size"), json_invalido=("json_valido", lambda s: f"{(~s).mean():.0%}"),
    tokens_entrada=("tokens_entrada", "sum"), tokens_salida=("tokens_salida", "sum"),
    latencia_p50_s=("latencia_s", "median"), latencia_p95_s=("latencia_s", lambda s: s.quantile(0.95)),
    costo_usd=("costo_usd", lambda s: s.sum(min_count=1)))
display(resumen)
gen = log[~log["tarea"].str.startswith("embeddings")]
print(f"Tasa global de JSON inválido: {(~gen['json_valido']).mean():.1%} (meta < 2 %)")
print(f"Latencia p95 de generación: {gen['latencia_s'].quantile(0.95):.1f} s (meta < 20 s)")
if log["costo_usd"].notna().any():
    print(f"Costo total registrado: USD {log['costo_usd'].sum():.4f}")
else:
    print("Costo: llena PRECIOS_USD_POR_MILLON en la configuración para calcularlo.")

# Métricas de aprendizaje (ilustrativas con la estudiante simulada)
fid = [e.fidelidad for c in CONTENIDO.values() for e in c["explicaciones"] if e.fidelidad is not None]
intentos = [p.intentos for p in ruta_viva.pasos if p.incluido and p.intentos]
primer = [c["calificaciones"][0].aprobado for c in CONTENIDO.values() if c["calificaciones"]]
temas_ruta_ids = [p.tema_id for p in ruta_viva.pasos if p.incluido]
qs_pre = [q for q in diagnostico.preguntas if q.tema_id in temas_ruta_ids]
pre = (sum(q.dificultad * aciertos[q.id] for q in qs_pre) / sum(q.dificultad for q in qs_pre)) if qs_pre else None
post = cal_final.puntaje_total if cal_final else None
metricas = {
    "fidelidad_RAG_tema_demo": fid[0] if fid else None,
    "temas_omitidos": sum(not p.incluido for p in ruta_viva.pasos),
    "aprobacion_primer_intento": f"{np.mean(primer):.0%}" if primer else None,
    "intentos_promedio_por_tema": round(float(np.mean(intentos)), 2) if intentos else None,
    "pre_test_temas_ruta": round(pre, 3) if pre is not None else None,
    "post_test_final": post,
    "ganancia_hake_g": round((post - pre) / (1 - pre), 3) if (pre is not None and post is not None and pre < 1) else None,
    "badge": BADGE is not None,
}
guardar("metricas_prueba", metricas)
display(pd.Series(metricas, name="valor").to_frame())
print("Nota: el pre y el post no usan ítems equivalentes; en el piloto deben construirse ítems paralelos para Hake.")

## 16. Paquete para el repositorio
Copia `mago_de_oz_out/schemas/` y `prompts/` a `prompts/` del repo, `salidas/` a `data/ejemplos/` y **guarda este notebook con sus salidas** en `notebooks/`: así cualquiera puede comprobar que el contenido de las pantallas salió del LLM y no fue redactado a mano.

In [ ]:
import shutil
zip_path = shutil.make_archive("mago_de_oz_smartmentor", "zip", OUT)
print("Paquete:", zip_path)
for carpeta in ["schemas", "prompts", "salidas"]:
    print(f"\n{carpeta}/:", ", ".join(sorted(p.name for p in (OUT / carpeta).iterdir())))
try:
    from google.colab import files
    files.download(zip_path)
except Exception:
    pass